# Real-Time Data Streaming Example

This notebook demonstrates how to use the BinanceStreamFetcher to collect real-time market data and integrate it with the trading system.

## Setup

First, let's import the necessary modules and initialize our stream fetcher.

In [ ]:
import os
import sys
import asyncio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import plotly.graph_objects as go
from IPython.display import clear_output
import time

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.stream_fetcher import BinanceStreamFetcher

## Initialize Stream Fetcher

Create an instance of BinanceStreamFetcher with our desired configuration.

In [ ]:
# Initialize fetcher
fetcher = BinanceStreamFetcher(
    symbol="BTCUSDT",
    channels=['kline_5m', 'trade', 'depth'],
    buffer_size=1000
)

## Real-Time Data Visualization

Create interactive visualizations of the streaming data.

In [ ]:
def create_candlestick_chart(kline_data):
    """Create candlestick chart using plotly."""
    fig = go.Figure(data=[
        go.Candlestick(
            x=kline_data.index,
            open=kline_data['open'],
            high=kline_data['high'],
            low=kline_data['low'],
            close=kline_data['close']
        )
    ])
    
    fig.update_layout(
        title='BTCUSDT Price',
        yaxis_title='Price (USDT)',
        xaxis_title='Time'
    )
    
    return fig

def create_order_book_chart(order_book):
    """Create order book visualization using plotly."""
    fig = go.Figure()
    
    # Add bids
    fig.add_trace(go.Scatter(
        x=order_book.index,
        y=order_book['bid_quantity'],
        fill='tozeroy',
        name='Bids'
    ))
    
    # Add asks
    fig.add_trace(go.Scatter(
        x=order_book.index,
        y=order_book['ask_quantity'],
        fill='tozeroy',
        name='Asks'
    ))
    
    fig.update_layout(
        title='Order Book Depth',
        xaxis_title='Price (USDT)',
        yaxis_title='Quantity (BTC)'
    )
    
    return fig

def create_trade_chart(trades):
    """Create trade visualization using plotly."""
    fig = go.Figure()
    
    # Add trades
    fig.add_trace(go.Scatter(
        x=trades.index,
        y=trades['price'],
        mode='markers',
        marker=dict(
            size=trades['quantity'] * 10,  # Scale marker size by quantity
            color=['red' if maker else 'green' for maker in trades['buyer_maker']]
        ),
        name='Trades'
    ))
    
    fig.update_layout(
        title='Recent Trades',
        xaxis_title='Time',
        yaxis_title='Price (USDT)'
    )
    
    return fig

## Start Data Streaming

Start the streaming process and display real-time updates.

In [ ]:
async def display_streaming_data():
    """Display streaming data with periodic updates."""
    try:
        # Start streaming in background
        stream_task = asyncio.create_task(fetcher.start())
        
        while True:
            # Get current data
            data = fetcher.get_current_data()
            
            # Clear previous output
            clear_output(wait=True)
            
            # Create and display visualizations
            if data['kline'] is not None:
                fig1 = create_candlestick_chart(data['kline'])
                fig1.show()
            
            if not data['order_book'].empty:
                fig2 = create_order_book_chart(data['order_book'])
                fig2.show()
            
            if not data['recent_trades'].empty:
                fig3 = create_trade_chart(data['recent_trades'])
                fig3.show()
            
            # Wait before next update
            await asyncio.sleep(1)
            
    except KeyboardInterrupt:
        await fetcher.stop()
        stream_task.cancel()
        try:
            await stream_task
        except asyncio.CancelledError:
            pass

In [ ]:
# Run the streaming visualization
# Note: Press Ctrl+C to stop
await display_streaming_data()

## Analyze Collected Data

After collecting some data, we can analyze it to extract insights.

In [ ]:
# Load collected data
today = datetime.now().strftime('%Y%m%d')
klines = pd.read_parquet(f'data/raw/btcusdt_klines_{today}.parquet')
trades = pd.read_parquet(f'data/raw/btcusdt_trades_{today}.parquet')
order_book = pd.read_parquet(f'data/raw/btcusdt_orderbook_{today}.parquet')

# Calculate basic statistics
print("Kline Statistics:")
print(klines['close'].describe())

print("\nTrade Statistics:")
print(f"Number of trades: {len(trades)}")
print(f"Average trade size: {trades['quantity'].mean():.4f} BTC")

print("\nOrder Book Statistics:")
print(f"Average bid-ask spread: {(order_book.index.max() - order_book.index.min()):.2f} USDT")
print(f"Total bid depth: {order_book['bid_quantity'].sum():.2f} BTC")
print(f"Total ask depth: {order_book['ask_quantity'].sum():.2f} BTC")

## Integration with Trading Strategy

Example of how to use the streaming data in a trading strategy.

In [ ]:
class SimpleStrategy:
    """Simple trading strategy example using streaming data."""
    
    def __init__(self, lookback_period: int = 20):
        self.lookback_period = lookback_period
        self.position = 0
        
    def calculate_signals(self, klines: pd.DataFrame) -> float:
        """Calculate trading signals from kline data."""
        if len(klines) < self.lookback_period:
            return 0
        
        # Calculate simple moving average
        sma = klines['close'].rolling(self.lookback_period).mean().iloc[-1]
        current_price = klines['close'].iloc[-1]
        
        # Generate signal
        if current_price > sma and self.position <= 0:
            return 1  # Buy signal
        elif current_price < sma and self.position >= 0:
            return -1  # Sell signal
        
        return 0  # Hold

async def run_strategy():
    """Run the trading strategy with streaming data."""
    strategy = SimpleStrategy()
    
    try:
        # Start streaming in background
        stream_task = asyncio.create_task(fetcher.start())
        
        while True:
            # Get current data
            data = fetcher.get_current_data()
            
            if data['kline'] is not None:
                # Calculate trading signals
                signal = strategy.calculate_signals(data['kline'])
                
                # Print strategy output
                print(f"Time: {datetime.now()}")
                print(f"Current Price: {data['kline']['close'].iloc[-1]:.2f}")
                print(f"Signal: {signal}")
                print("-" * 50)
            
            # Wait before next update
            await asyncio.sleep(5)
            
    except KeyboardInterrupt:
        await fetcher.stop()
        stream_task.cancel()
        try:
            await stream_task
        except asyncio.CancelledError:
            pass

In [ ]:
# Run the trading strategy
# Note: Press Ctrl+C to stop
await run_strategy()